# 01 - Exploration des Données

Ce notebook explore les données vaccinales pour comprendre les patterns et préparer le feature engineering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Configuration
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully")

## 1. Chargement des données

In [ ]:
# Chargement des données (à adapter selon la source)
# Pour l'instant, nous allons simuler des données

def generate_sample_data(n_enfants=1000):
    """Génère des données simulées pour l'exploration"""
    np.random.seed(42)
    
    # Données démographiques
    data = {
        'enfant_id': [f'enfant_{i:04d}' for i in range(n_enfants)],
        'date_naissance': [datetime.now() - timedelta(days=np.random.randint(30, 1825)) for _ in range(n_enfants)],
        'sexe': np.random.choice(['M', 'F'], n_enfants),
        'region': np.random.choice(['Dakar', 'Thiès', 'Kaolack', 'Saint-Louis', 'Diourbel'], n_enfants),
        'centre_sante': [f'Centre_{np.random.randint(1, 11)}' for _ in range(n_enfants)],
        'nombre_vaccinations': np.random.poisson(3, n_enfants),
        'derniere_vaccination': [datetime.now() - timedelta(days=np.random.randint(1, 365)) for _ in range(n_enfants)],
        'retard_vaccinal': np.random.choice([0, 1], n_enfants, p=[0.7, 0.3])  # 30% en retard
    }
    
    return pd.DataFrame(data)

# Générer les données
df = generate_sample_data()
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Analyse descriptive

In [ ]:
# Statistiques descriptives
print("Statistiques descriptives:")
df.describe(include='all')

In [ ]:
# Distribution des retards vaccinaux
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
df['retard_vaccinal'].value_counts().plot(kind='bar', color=['green', 'red'])
plt.title('Distribution des retards vaccinaux')
plt.xlabel('Retard (0=Non, 1=Oui)')
plt.ylabel('Nombre d\'enfants')

plt.subplot(1, 2, 2)
df['retard_vaccinal'].value_counts(normalize=True).plot(kind='pie', autopct='%1.1f%%')
plt.title('Pourcentage de retards vaccinaux')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution par région
plt.figure(figsize=(10, 6))

retard_par_region = df.groupby('region')['retard_vaccinal'].mean().sort_values(ascending=False)
retard_par_region.plot(kind='bar', color='coral')
plt.title('Taux de retard vaccinal par région')
plt.xlabel('Région')
plt.ylabel('Taux de retard')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution du nombre de vaccinations
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
df['nombre_vaccinations'].hist(bins=20, alpha=0.7, color='skyblue')
plt.title('Distribution du nombre de vaccinations')
plt.xlabel('Nombre de vaccinations')
plt.ylabel('Fréquence')

plt.subplot(1, 2, 2)
sns.boxplot(x='retard_vaccinal', y='nombre_vaccinations', data=df)
plt.title('Nombre de vaccinations vs Retard')
plt.xlabel('Retard vaccinal')
plt.ylabel('Nombre de vaccinations')

plt.tight_layout()
plt.show()

## 3. Feature Engineering initial

In [ ]:
# Création de features temporels
df['age_jours'] = (datetime.now() - df['date_naissance']).dt.days
df['age_mois'] = df['age_jours'] // 30
df['jours_derniere_vaccination'] = (datetime.now() - df['derniere_vaccination']).dt.days

# Features de vaccination
df['vaccinations_par_mois'] = df['nombre_vaccinations'] / (df['age_mois'] + 1)
df['intervalle_moyen_vaccinations'] = df['jours_derniere_vaccination'] / (df['nombre_vaccinations'] + 1)

# Features catégorielles encodées
df['sexe_encoded'] = (df['sexe'] == 'M').astype(int)

print("Features créées avec succès")
df[['age_mois', 'jours_derniere_vaccination', 'vaccinations_par_mois', 'sexe_encoded']].head()

## 4. Analyse des corrélations

In [ ]:
# Matrice de corrélation
numeric_features = ['age_mois', 'nombre_vaccinations', 'jours_derniere_vaccination', 
                   'vaccinations_par_mois', 'intervalle_moyen_vaccinations', 'sexe_encoded', 'retard_vaccinal']

correlation_matrix = df[numeric_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f')
plt.title('Matrice de corrélation')
plt.tight_layout()
plt.show()

In [ ]:
# Corrélations avec la variable cible
correlations_with_target = correlation_matrix['retard_vaccinal'].sort_values(ascending=False)
print("Corrélations avec le retard vaccinal:")
print(correlations_with_target)

## 5. Conclusions de l'exploration

### Points clés identifiés:
1. **Distribution déséquilibrée**: ~30% des enfants sont en retard vaccinal
2. **Facteurs géographiques**: Certaines régions montrent des taux plus élevés
3. **Features temporelles importantes**: Âge et intervalle depuis dernière vaccination
4. **Corrélations modérées**: Aucune corrélation très forte, suggérant la nécessité d'un modèle complexe

### Prochaines étapes:
1. Feature engineering plus avancé
2. Création de features d'interaction
3. Validation des features avec les experts du domaine
4. Préparation pour le modèle de machine learning